
# `hyprat`: Exact Rational Hypercomplex Numbers

This notebook is a guided, hands-on tour of the **`hyprat`** package: a
single immutable class, `Hy`, that represents **exact, rational-valued
hypercomplex numbers** of arbitrary rank -- reals, complex numbers,
quaternions, octonions, sedenions, and beyond -- built recursively via the
[Cayley&ndash;Dickson construction](https://en.wikipedia.org/wiki/Cayley%E2%80%93Dickson_construction).

## What is the Cayley&ndash;Dickson construction?

Starting from the real numbers $\mathbb{R}$, the Cayley&ndash;Dickson
construction repeatedly *doubles* an algebra $A$ into a new algebra
$A^2 = A \times A$ by defining, for pairs $(a, b)$ and $(c, d)$ with
$a,b,c,d \in A$:

$$
(a, b) + (c, d) = (a + c,\; b + d)
$$
$$
(a, b)(c, d) = (ac - \overline{d}\,b,\; da + b\,\overline{c})
$$
$$
\overline{(a, b)} = (\overline{a}, -b)
$$

where $\overline{a}$ denotes the conjugate of $a$ (with $\overline{a} = a$
for real $a$). Applying this once to $\mathbb{R}$ gives $\mathbb{C}$;
applying it again gives the quaternions $\mathbb{H}$; again gives the
octonions $\mathbb{O}$; again gives the sedenions $\mathbb{S}$; and so on
forever, with the dimension doubling ($1, 2, 4, 8, 16, \dots$) at every
step. Each doubling costs the algebra an algebraic property:

| Algebra              | Rank | Dim | Commutative? | Associative? | Division algebra (no zero divisors)? |
|----------------------|:----:|:---:|:-------------:|:-------------:|:--------------------------------------:|
| Reals $\mathbb{R}$     | 0 | 1  | yes | yes | yes |
| Complex $\mathbb{C}$   | 1 | 2  | yes | yes | yes |
| Quaternions $\mathbb{H}$ | 2 | 4  | **no** | yes | yes |
| Octonions $\mathbb{O}$   | 3 | 8  | no | **no** (only *alternative*) | yes |
| Sedenions $\mathbb{S}$   | 4 | 16 | no | no | **no** |

`hyprat`'s `Hy` class implements exactly this tower, using Python's exact
`fractions.Fraction` for every real coordinate -- so every result is
*exact*, with no floating-point rounding, no matter how deep the tower
goes.

## The `Hy` representation

Every `Hy` value, of every rank, stores exactly **two** components,
`.real` and `.imag`, which are always the same "shape":

```
rank 0  ->  a plain fractions.Fraction               (a "real")
rank 1  ->  Hy(real, imag), real/imag : Fraction      (a "complex" number)
rank 2  ->  Hy(h1, h2),     h1/h2     : rank-1 Hy     (a "quaternion")
rank 3  ->  Hy(h3, h4),     h3/h4     : rank-2 Hy     (an "octonion")
rank n  ->  Hy(x, y),       x/y       : rank-(n-1) Hy
```

The constructor automatically promotes ("embeds") whatever you pass it so
that this invariant always holds -- you'll see plenty of examples below.

Let's dive in.



## 0. Setup

If you haven't installed `hyprat` yet:

```bash
pip install git+https://github.com/alreich/hyper_rationals.git
```

For this notebook, we just need the single public name `Hy`.


In [1]:

from hyprat import Hy
from fractions import Fraction

print("hyprat.Hy imported OK")


hyprat.Hy imported OK



## 1. Rank 0 & rank 1: rationals and complex numbers

A bare number given to `Hy(...)` becomes a **rank-1** value (a "complex"
number) with an implicit zero imaginary part -- exactly the way Python's
own `complex(3)` is `(3+0j)`.

`Hy` accepts `int`, `float`, `Fraction`, `str` (including fraction
strings like `'5/2'` and decimal strings like `'3.2'`), and even Python's
built-in `complex`, for either argument. Two more ways to build a value:
`Hy.from_array([...])`, from a flat list of coefficients, and
`Hy.random(rank)`, which draws one at random -- both covered in more
detail as we go.


In [2]:

# A variety of equivalent ways to build the same rank-1 value:
print(Hy(3))                 # int
print(Hy('5/2'))             # a fraction string
print(Hy(2.5))               # a float
print(Hy(Fraction(5, 2)))    # a Fraction directly

# Two arguments -> real, imag:
z = Hy('5/2', '-16/5')
print("z       =", z)
print("z.real  =", z.real, " z.imag =", z.imag)
print("z.rank  =", z.rank, " z.dimension =", z.dimension)

# ...or from a flat list of its 2**rank coefficients (numbers and fraction
# strings can be freely mixed):
print(Hy.from_array(['5/2', -3.2]))

# ...or just draw a random one, with a seed for reproducibility:
print(Hy.random(1, seed=0))


(3)
(5/2)
(5/2)
(5/2)
z       = (5/2-16/5j)
z.real  = 5/2  z.imag = -16/5
z.rank  = 1  z.dimension = 2
(5/2-16/5j)
(3/4-8/3j)


In [3]:

# Floats are interpreted via their "obvious" decimal value, not the exact
# (ugly) binary value that a naive Fraction(float) would give:
print(Fraction(3.2))          # the ugly exact-binary-value fraction
print(Hy(3.2).real)           # hyprat gives you the friendly 16/5 instead


3602879701896397/1125899906842624
16/5


In [4]:

# Hy plays nicely with Python's built-in complex type, both for
# construction and comparison:
z = Hy('5/2', '-16/5')
print(z == complex(2.5, -3.2))     # True -- cross-type equality
print(complex(z))                   # convert back to a Python complex

w = Hy(complex(1, 1))                # construct directly from a complex
print(w)


True
(2.5-3.2j)
(1+j)


In [5]:

# Ordinary arithmetic, exactly, with no floating-point error:
a = Hy('1/3', '1/7')
b = Hy('2/3', '-1/7')
print("a + b =", a + b)
print("a - b =", a - b)
print("a * b =", a * b)
print("a / b =", a / b)

# Hy also happily mixes with plain Python numbers on either side:
print(a + 1, 1 - a, 2 * a, a / 2)


a + b = (1)
a - b = (-1/3+2/7j)
a * b = (107/441+1/21j)
a / b = (89/205+63/205j)
(4/3+1/7j) (2/3-1/7j) (2/3+2/7j) (1/6+1/14j)



## 2. Rank 2: quaternions

A **quaternion** is built by pairing up two rank-1 (complex) `Hy` values:
`Hy(h1, h2)` where `h1, h2` are each rank-1. The result has rank 2 and
4 real coordinates: $a + bi + cj + dk$.

Quaternion multiplication is **associative but not commutative** -- the
famous relations $ij = k$, $ji = -k$, etc.


In [6]:

one = Hy(1)
i = Hy(Hy(0, 1), Hy(0, 0))
j = Hy(Hy(0, 0), Hy(1, 0))
k = Hy(Hy(0, 0), Hy(0, 1))

for name, val in [("1", one), ("i", i), ("j", j), ("k", k)]:
    print(f"{name} = {val}   (rank {val.rank}, dim {val.dimension})")


1 = (1)   (rank 1, dim 2)
i = (i)   (rank 2, dim 4)
j = (j)   (rank 2, dim 4)
k = (k)   (rank 2, dim 4)


In [7]:

# The defining quaternion multiplication table:
print("i*i =", i * i, "   j*j =", j * j, "   k*k =", k * k)
print("i*j =", i * j, "   j*i =", j * i, "   <-- non-commutative!")
print("j*k =", j * k, "   k*j =", k * j)
print("k*i =", k * i, "   i*k =", i * k)

assert i * j == k and j * i == -k
assert (i * j) * k == i * (j * k)  # associativity DOES hold for quaternions
print("\nAssociativity check ((ij)k == i(jk)):", (i * j) * k == i * (j * k))


i*i = (-1)    j*j = (-1)    k*k = (-1)
i*j = (k)    j*i = (-k)    <-- non-commutative!
j*k = (i)    k*j = (-i)
k*i = (j)    i*k = (-j)

Associativity check ((ij)k == i(jk)): True


In [8]:

# A general quaternion, its conjugate, norm, and inverse -- built here
# straight from a flat array of its 4 coefficients:
q = Hy.from_array([1, 2, 3, 4])     # 1 + 2i + 3j + 4k
print("q          =", q)
print("q.to_array() ->", q.to_array(), " (the inverse of from_array())")
print("conj(q)    =", q.conjugate())
print("norm(q)    =", q.norm(), " (== 1^2+2^2+3^2+4^2 =", 1+4+9+16, ")")
print("q^-1       =", q.inverse())
print("q * q^-1   =", q * q.inverse(), " (should be 1)")


q          = (1+2i+3j+4k)
q.to_array() -> [Fraction(1, 1), Fraction(2, 1), Fraction(3, 1), Fraction(4, 1)]  (the inverse of from_array())
conj(q)    = (1-2i-3j-4k)
norm(q)    = 30  (== 1^2+2^2+3^2+4^2 = 30 )
q^-1       = (1/30-1/15i-1/10j-2/15k)
q * q^-1   = (1)  (should be 1)



## 3. Rank 3: octonions

An **octonion** pairs two quaternions: `Hy(h3, h4)` with `h3, h4` rank-2.
Octonions have 8 real coordinates and lose **associativity** -- but they
remain an *alternative* algebra, meaning any sub-algebra generated by just
two elements *is* associative. Concretely:

$$x(xy) = (xx)y \qquad\text{and}\qquad (yx)x = y(xx)$$

always hold, even though $x(yz) \ne (xy)z$ in general.


In [9]:

h3 = Hy(Hy(1, 0), Hy(0, 0))
h4 = Hy(Hy(0, 1), Hy(0, 0))
o = Hy(h3, h4)
print("o =", o, "  rank =", o.rank, "  dimension =", o.dimension)
print("o components:", o.components())
print("o as a flat array:", o.to_array(as_str=True))


o = (1+iL)   rank = 3   dimension = 8
o components: (Fraction(1, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(1, 1), Fraction(0, 1), Fraction(0, 1))
o as a flat array: ['1', '0', '0', '0', '0', '1', '0', '0']


In [10]:

# Octonions are generally non-associative. Rather than hand-writing three
# example values, let's just draw them at random -- Hy.random(rank, seed=...)
# gives a reproducible random value of any rank, handy for exactly this
# kind of throwaway example (and for the fuzz-testing style examples later
# in this notebook):
x = Hy.random(3, seed=1)
y = Hy.random(3, seed=2)
z = Hy.random(3, seed=3)

lhs, rhs = (x * y) * z, x * (y * z)
print("x       =", x)
print("y       =", y)
print("z       =", z)
print("(xy)z   =", lhs)
print("x(yz)   =", rhs)
print("equal?  ", lhs == rhs, " <-- associativity fails in general for octonions")


x       = (-1-7/3i-3/2j+5/4k+3/2L-3/2iL-9/4jL+4/5kL)
y       = (-8-7/3i-2/3j-3/5L-8/5iL-jL+1/2kL)
z       = (-2/5+4i+2/5j+k+9L-9/4iL-1/5jL-kL)
(xy)z   = (740327/9000-1348769/12000i-328771/1125j-86021/3000k+3341861/36000L+3632917/18000iL+199481/3600jL+9449/720kL)
x(yz)   = (740327/9000-1608551/12000i-1255609/4500j+42033/1000k+3086419/36000L+3432473/18000iL+327359/3600jL-156191/3600kL)
equal?   False  <-- associativity fails in general for octonions


In [11]:

# ...but they ARE alternative: x(xy) == (xx)y and (yx)x == y(xx) always hold.
print("x(xy) == (xx)y ?", x * (x * y) == (x * x) * y)
print("(yx)x == y(xx) ?", (y * x) * x == y * (x * x))


x(xy) == (xx)y ? True
(yx)x == y(xx) ? True



## 4. Rank 4+: sedenions and beyond

One more doubling gives the **sedenions** (16 real coordinates, rank 4).
Here the algebra loses its *last* nice property: sedenions have
**zero divisors** -- nonzero $u, v$ with $uv = 0$ -- so they're not even a
division algebra anymore. `hyprat` doesn't stop you here: the same
Cayley&ndash;Dickson formulas keep working at every rank, they just stop
guaranteeing certain algebraic properties.


In [12]:

# Build the 16 sedenion basis units e_0 (=1), e_1, ..., e_15 via from_array
# -- a flat list of 15 zeros and a single 1 is an easy way to name a basis
# vector by index:
basis = [Hy.from_array([1 if k == idx else 0 for k in range(16)]) for idx in range(16)]

# A known example of sedenion zero divisors: (e_a+e_b)(e_c-e_d) == 0 for
# certain index combinations (the exact indices are convention-dependent,
# so we search for one rather than hard-coding a published example):
zero = Hy.from_array([0] * 16)
found = None
for a_idx in range(1, 16):
    for b_idx in range(a_idx + 1, 16):
        u = basis[a_idx] + basis[b_idx]
        for c_idx in range(1, 16):
            for d_idx in range(c_idx + 1, 16):
                if {c_idx, d_idx} == {a_idx, b_idx}:
                    continue
                v = basis[c_idx] - basis[d_idx]
                if u * v == zero:
                    found = (u, v)
                    break
            if found:
                break
        if found:
            break
    if found:
        break

u, v = found
print("u =", u)
print("v =", v)
print("u * v =", u * v, " <-- zero, even though both u and v are nonzero!")


u = (e1+e10)
v = (e4-e15)
u * v = (0)  <-- zero, even though both u and v are nonzero!


In [13]:

# hyprat doesn't limit you to sedenions -- rank 5 ("pathions", dim 32),
# rank 6 ("chingons", dim 64), etc. all work identically. Hy.random(rank)
# is the easiest way to get your hands on one:
p = Hy.random(5, seed=42)
print("A random rank-5 value ('pathion'):")
print("  rank =", p.rank, " dimension =", p.dimension)
print("  as an array:", p.to_array())


A random rank-5 value ('pathion'):
  rank = 5  dimension = 32
  as an array: [Fraction(-6, 1), Fraction(-1, 2), Fraction(-1, 1), Fraction(-1, 1), Fraction(8, 1), Fraction(9, 4), Fraction(-8, 1), Fraction(-7, 2), Fraction(-2, 5), Fraction(-9, 5), Fraction(-1, 2), Fraction(2, 1), Fraction(-1, 2), Fraction(3, 1), Fraction(-9, 2), Fraction(4, 3), Fraction(-1, 2), Fraction(-1, 1), Fraction(-6, 1), Fraction(3, 1), Fraction(2, 3), Fraction(-1, 1), Fraction(1, 1), Fraction(-3, 2), Fraction(-7, 5), Fraction(0, 1), Fraction(2, 5), Fraction(-1, 2), Fraction(-7, 1), Fraction(-2, 3), Fraction(-7, 2), Fraction(-3, 2)]



## 5. Common API at a glance

Regardless of rank, every `Hy` supports the same core API:


In [14]:

q = Hy(Hy(1, 2), Hy(3, 4))

print("q.real           ->", q.real)
print("q.imag           ->", q.imag)
print("q.rank           ->", q.rank)
print("q.dimension      ->", q.dimension)
print("q.components()   ->", q.components())
print("q.to_array()     ->", q.to_array(), " (like components(), but a plain list)")
print("q.to_array(as_str=True) ->", q.to_array(as_str=True), " (as strings, e.g. for JSON)")
print("q.conjugate()    ->", q.conjugate())
print("q.norm()         ->", q.norm())
print("q.inverse()      ->", q.inverse())
print("q.is_zero()      ->", q.is_zero())
print("abs(q)           ->", abs(q), " (a float, since the exact norm is generally irrational)")
print("bool(q)          ->", bool(q))
print("q ** 2           ->", q ** 2)
print("q ** -1          ->", q ** -1, " (== q.inverse())")
print("list(q)          ->", list(q), " (iterates real, imag)")
print("q[0], q[1]       ->", q[0], ",", q[1])


q.real           -> (1+2j)
q.imag           -> (3+4j)
q.rank           -> 2
q.dimension      -> 4
q.components()   -> (Fraction(1, 1), Fraction(2, 1), Fraction(3, 1), Fraction(4, 1))
q.to_array()     -> [Fraction(1, 1), Fraction(2, 1), Fraction(3, 1), Fraction(4, 1)]  (like components(), but a plain list)
q.to_array(as_str=True) -> ['1', '2', '3', '4']  (as strings, e.g. for JSON)
q.conjugate()    -> (1-2i-3j-4k)
q.norm()         -> 30
q.inverse()      -> (1/30-1/15i-1/10j-2/15k)
q.is_zero()      -> False
abs(q)           -> 5.477225575051661  (a float, since the exact norm is generally irrational)
bool(q)          -> True
q ** 2           -> (-28+4i+6j+8k)
q ** -1          -> (1/30-1/15i-1/10j-2/15k)  (== q.inverse())
list(q)          -> [Hy('1', '2'), Hy('3', '4')]  (iterates real, imag)
q[0], q[1]       -> (1+2j) , (3+4j)



### String forms: `str`, `repr`, and `Hy.parse` / `Hy.from_string`

`str()` renders a `Hy` the way Python renders `complex` for rank 1
(`a+bj`), the customary `a+bi+cj+dk` for rank 2, `i, j, k, L, iL, jL,
kL` labelled basis units for rank 3 ("octonions"), and `e1, e2, ...`
labelled basis units for rank &ge; 4. `Hy.parse` (aliased as
`Hy.from_string`) is the exact inverse.


In [15]:

values = [
    Hy('5/2', '-16/5'),
    Hy(Hy(1, 2), Hy(3, 4)),
    Hy(Hy(Hy(1, 0), Hy(0, 0)), Hy(Hy(0, 1), Hy(0, 0))),
]
for v in values:
    s = str(v)
    r = repr(v)
    parsed = Hy.parse(s)
    print(f"str  = {s!r:20}  repr = {r:35}  parse(str) == v: {parsed == v}")


str  = '(5/2-16/5j)'         repr = Hy('5/2', '-16/5')                   parse(str) == v: True
str  = '(1+2i+3j+4k)'        repr = Hy(Hy('1', '2'), Hy('3', '4'))       parse(str) == v: True
str  = '(1+iL)'              repr = Hy(Hy(Hy('1', '0'), Hy('0', '0')), Hy(Hy('0', '1'), Hy('0', '0')))  parse(str) == v: True



### Units and `is_unit()`

Every rank has a set of "unit" values -- `+-1`, plus the `+-1` basis
elements like `j`, or `i`/`j`/`k`, or `i, j, k, L, iL, jL, kL` (rank 3),
or `e1, e2, ...` (rank &ge; 4) -- each with exactly one real coordinate
equal to `+-1` and every other coordinate 0.
`Hy.units(rank)` returns all of them for a given rank, as a dict keyed
by their string form; `some_hy.is_unit()` answers the same question for
a single value:


In [16]:

print("Hy.units(1)  ->", Hy.units(1))
print("Hy.units(2)  ->", Hy.units(2))
print("Hy.units(3) has", len(Hy.units(3)), "entries, e.g.:", {k: Hy.units(3)[k] for k in ("1", "iL", "-kL")})

print()
print("i.is_unit()      ->", i.is_unit())
print("(i + j).is_unit()->", (i + j).is_unit())
print("Hy(0, 0).is_unit() ->", Hy(0, 0).is_unit())


Hy.units(1)  -> {'1': Hy('1', '0'), '-1': Hy('-1', '0'), 'j': Hy('0', '1'), '-j': Hy('0', '-1')}
Hy.units(2)  -> {'1': Hy(Hy('1', '0'), Hy('0', '0')), '-1': Hy(Hy('-1', '0'), Hy('0', '0')), 'i': Hy(Hy('0', '1'), Hy('0', '0')), '-i': Hy(Hy('0', '-1'), Hy('0', '0')), 'j': Hy(Hy('0', '0'), Hy('1', '0')), '-j': Hy(Hy('0', '0'), Hy('-1', '0')), 'k': Hy(Hy('0', '0'), Hy('0', '1')), '-k': Hy(Hy('0', '0'), Hy('0', '-1'))}
Hy.units(3) has 16 entries, e.g.: {'1': Hy(Hy(Hy('1', '0'), Hy('0', '0')), Hy(Hy('0', '0'), Hy('0', '0'))), 'iL': Hy(Hy(Hy('0', '0'), Hy('0', '0')), Hy(Hy('0', '1'), Hy('0', '0'))), '-kL': Hy(Hy(Hy('0', '0'), Hy('0', '0')), Hy(Hy('0', '0'), Hy('0', '-1')))}

i.is_unit()      -> True
(i + j).is_unit()-> False
Hy(0, 0).is_unit() -> False



### LaTeX rendering: `.latex()`

`some_hy.latex()` renders a value as a LaTeX math expression, which
Jupyter can display as typeset math via `IPython.display.Math`.
Basis labels at rank 1-3 (`j`; `i, j, k`; `i, j, k, L, iL, jL, kL`)
render unchanged, while `e1, e2, ...` labels from rank 4 up are
subscripted (`e_{1}`, `e_{2}`, ...); non-integer coefficients are
rendered with a horizontal fraction bar by default -- pass
`vinculum='diagonal'` for a plain slash instead:


In [17]:

from IPython.display import Math, display

z = Hy('5/2', '-16/5')
print("z.latex()                    ->", z.latex())
print("z.latex(vinculum='diagonal') ->", z.latex(vinculum='diagonal'))
print("o.latex()                    ->", o.latex(), " (subscripted e-labels)")

display(Math(z.latex()))
display(Math(z.latex(vinculum='diagonal')))
display(Math(q.latex()))

# mode= wraps the expression in LaTeX math delimiters, if you need the
# delimiters included rather than adding them yourself:
print("z.latex(mode='inline')  ->", z.latex(mode='inline'))
print("z.latex(mode='display') ->", z.latex(mode='display'))


z.latex()                    -> \frac{5}{2}-\frac{16}{5}j
z.latex(vinculum='diagonal') -> 5/2-16/5j
o.latex()                    -> 1+iL  (subscripted e-labels)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

z.latex(mode='inline')  -> $\frac{5}{2}-\frac{16}{5}j$
z.latex(mode='display') -> \[\frac{5}{2}-\frac{16}{5}j\]



### Random values: `Hy.random()` and reproducibility

`Hy.random(rank)`, used throughout this notebook to conjure up example
values, draws each of a value's `2**rank` coefficients as an independent
random `Fraction(n, d)`, with `n` uniform in `[lo, hi]` (default
`[-9, 9]`) and `d` uniform in `[1, dmax]` (default `[1, 6]`). There are a
few ways to control reproducibility:


In [18]:

# A one-off seed=, scoped to just this call:
a = Hy.random(2, seed=7)
b = Hy.random(2, seed=7)
print("Hy.random(2, seed=7) is reproducible:", a == b, " ->", a)

# Hy.seed(...) instead fixes a shared default RNG for everything that
# follows, so bare Hy.random(rank) calls become reproducible too:
Hy.seed(2026)
run1 = [Hy.random(2) for _ in range(3)]
Hy.seed(2026)
run2 = [Hy.random(2) for _ in range(3)]
print("Two Hy.seed(2026)-seeded runs match:", run1 == run2)

# Or bring your own random.Random for full control / sharing across calls:
import random
my_rng = random.Random(123)
c1 = Hy.random(2, rng=my_rng)
c2 = Hy.random(2, rng=my_rng)   # draws the *next* values from the same stream
print("Two draws from a shared rng (generally different):", c1, c2)

# You can also narrow the coefficient ranges with lo=/hi=/dmax=:
small = Hy.random(2, lo=-1, hi=1, dmax=2, seed=0)
print("Small-coefficient quaternion:", small)

# rank must be a positive int -- rank 0 isn't a valid Hy (every Hy has
# rank >= 1 by construction):
try:
    Hy.random(0)
except ValueError as e:
    print("Hy.random(0) raised ValueError:", e)


Hy.random(2, seed=7) is reproducible: True  -> (1/2+1/2i-8j+8k)
Two Hy.seed(2026)-seeded runs match: True
Two draws from a shared rng (generally different): (-8/3-7/4i-j-2k) (8/5+1/3i-4j-5/3k)
Small-coefficient quaternion: (-1/2i+1/2j)
Hy.random(0) raised ValueError: rank must be a positive int, got 0



### Flat-array conversion: `Hy.from_array()` and `.to_array()`

We've been using `Hy.from_array()`/`.to_array()` throughout this notebook
as a convenient way to build and inspect values. To recap: `to_array()`
flattens a `Hy` into a plain Python list of its `2**rank` coordinates (as
`Fraction`s, or as strings with `as_str=True`); `from_array()` is the
inverse, and happily accepts `int`, `float`, `Fraction`, and
plain-fraction strings like `'5/2'`, freely mixed in the same call. The
length must be a power of 2 that's at least 2 (every `Hy` has rank &ge;
1, so there's no length-1 case). This pair is convenient for interop with
things like `numpy` arrays, or for text-based serialization:


In [19]:

# Numbers, strings, and a mix of both -- all equivalent ways in:
q1 = Hy.from_array([1, 2, 3, 4])
q2 = Hy.from_array(['1', '2', '3', '4'])
q3 = Hy.from_array([1, '2/3', 3.5, Fraction(-1, 4)])
print(q1, "==", q2, "?", q1 == q2)
print("q3 =", q3)

# A round trip through JSON, via as_str=True:
import json
o = Hy.random(3, seed=99)
payload = json.dumps(o.to_array(as_str=True))
print("Serialized octonion:", payload)
restored = Hy.from_array(json.loads(payload))
print("Restored == original:", restored == o)


(1+2i+3j+4k) == (1+2i+3j+4k) ? True
q3 = (1+2/3i+7/2j-1/4k)
Serialized octonion: ["3/4", "-3/5", "-2", "-1", "-7/3", "3/5", "8", "3"]
Restored == original: True


In [20]:

# Error cases: the length must be a power of 2 (>= 2)...
for bad in ([], [1], [1, 2, 3]):
    try:
        Hy.from_array(bad)
    except ValueError as e:
        print(f"Hy.from_array({bad!r}) raised ValueError: {e}")

# ...and each element must be a *plain* fraction/decimal, not a composite
# expression like '1+2j' (use Hy.parse()/Hy(str) for those instead):
try:
    Hy.from_array(['1+2j', '3'])
except ValueError as e:
    print("Hy.from_array(['1+2j', '3']) raised ValueError:", e)


Hy.from_array([]) raised ValueError: from_array() requires a length that is a power of 2 and at least 2; got 0
Hy.from_array([1]) raised ValueError: from_array() requires a length that is a power of 2 and at least 2; got 1
Hy.from_array([1, 2, 3]) raised ValueError: from_array() requires a length that is a power of 2 and at least 2; got 3
Hy.from_array(['1+2j', '3']) raised ValueError: cannot parse '1+2j' as a plain fraction for from_array()



## 6. Putting it together: empirically verifying algebraic laws

A nice way to build confidence in an implementation like this is to
generate many random values and check that the properties we *expect* to
hold (or fail!) at each rank actually do -- this is exactly the strategy
`hyprat`'s own test suite uses, and `Hy.random()` makes it easy to write.


In [21]:

def check_norm_is_multiplicative(rank, trials=200, seed=0):
    "|xy| == |x||y| should hold for ranks 0-3 (reals/complex/quaternions/octonions)."
    ok = True
    for t in range(trials):
        x = Hy.random(rank, seed=f"{seed}-{rank}-{t}-x")
        y = Hy.random(rank, seed=f"{seed}-{rank}-{t}-y")
        if (x * y).norm() != x.norm() * y.norm():
            ok = False
            break
    return ok

for rank in (1, 2, 3):
    print(f"rank {rank}: norm multiplicative over 200 random trials? ", check_norm_is_multiplicative(rank))


rank 1: norm multiplicative over 200 random trials?  True
rank 2: norm multiplicative over 200 random trials?  True
rank 3: norm multiplicative over 200 random trials?  True


In [22]:

import random as _random

def find_a_sedenion_zero_divisor_via_random_basis_search(trials=2000, seed=0):
    "Randomly sample sums/differences of sedenion basis units until we hit a zero divisor."
    rng = _random.Random(seed)
    zero = Hy.from_array([0] * 16)
    for t in range(trials):
        i1, i2, i3, i4 = (rng.randint(1, 15) for _ in range(4))
        if len({i1, i2, i3, i4}) < 4:
            continue
        u = basis[i1] + basis[i2]
        v = basis[i3] - basis[i4]
        if (u * v) == zero:
            return u, v, t
    return None

result = find_a_sedenion_zero_divisor_via_random_basis_search()
if result:
    u, v, t = result
    print(f"Found a zero-divisor pair after {t + 1} random trials:")
    print("  u =", u)
    print("  v =", v)
    print("  u * v =", u * v)
else:
    print("No zero divisor found in this many trials (try increasing `trials`).")


Found a zero-divisor pair after 17 random trials:
  u = (e1+e15)
  v = (e2-e12)
  u * v = (0)


In [23]:

def check_associativity_failure_rate(rank, trials=500, seed=0):
    "How often does random (xy)z land exactly on x(yz)? Should be common for"
    "quaternions (rank 2, always associative) and rare for octonions (rank 3)."
    hits = 0
    for t in range(trials):
        x = Hy.random(rank, seed=f"{seed}-{rank}-{t}-x")
        y = Hy.random(rank, seed=f"{seed}-{rank}-{t}-y")
        z = Hy.random(rank, seed=f"{seed}-{rank}-{t}-z")
        if (x * y) * z == x * (y * z):
            hits += 1
    return hits, trials

for rank, label in [(2, "quaternions"), (3, "octonions")]:
    hits, trials = check_associativity_failure_rate(rank)
    print(f"{label:12} (rank {rank}): associative on {hits}/{trials} random triples")


quaternions  (rank 2): associative on 500/500 random triples


octonions    (rank 3): associative on 0/500 random triples



## Summary

* `Hy` is a single, immutable class representing exact rational
  hypercomplex numbers at *any* Cayley&ndash;Dickson rank: reals (0),
  complex (1), quaternions (2), octonions (3), sedenions (4), and beyond.
* Standard arithmetic (`+ - * /`), conjugation, norm, and inversion all
  follow the classical recursive formulas, using exact `Fraction`
  arithmetic throughout.
* `str()`/`repr()`/`Hy.parse()` (a.k.a. `Hy.from_string()`) give you
  human-readable round-trip string forms.
* `Hy.random(rank, ...)` generates random values of any rank, with
  reproducibility controlled via `Hy.seed(...)`, or per-call
  `seed=`/`rng=` keywords.
* `Hy.from_array([...])` / `some_hy.to_array(as_str=...)` convert to and
  from flat lists of coefficients -- numbers, fraction strings, or a mix
  of both -- for easy interop and serialization.

See the package's [GitHub repository](https://github.com/alreich/hyper_rationals)
and [Read the Docs](https://hyper-rationals.readthedocs.io/) page for the
full API reference and source.
